# Foundations of LLM Agentic Systems
### Structured study notes based on the supplied 10-page PDF

**Scope:** These notes follow the source presentation's concepts and terminology. Explanatory examples are added to make the ideas easier to apply; they are illustrative, not quoted from the PDF.

## Learning objectives
By the end, you should be able to:
- Explain why an agent is more than an LLM generating text.
- Trace the ReAct-style decide–execute–observe loop.
- Identify common agent actions and apply least-privilege boundaries.
- Explain how MCP standardizes tool access.
- Choose between sequential, parallel, handoff, group-chat, and Magentic coordination.
- Distinguish classic RAG from Agent RAG.
- Design explicit stop conditions and failure paths.

## 1. What is an LLM agent?
**Source: PDF pages 2–3**

An agent is not simply a model producing text. It is an LLM operating inside an **orchestration layer** that turns model decisions into actions.

The core responsibility split is:

- **Model / agent:** interprets context and chooses what should happen next.
- **Orchestration layer:** validates and executes the requested action, captures the result, adds it to context, and decides whether to continue or stop.

> **Key principle:** The model chooses; the orchestration layer executes.

### 1.1 The ReAct-style loop
**Source: PDF page 2**

1. **Read context** — task, instructions, previous messages, tool results, and other relevant information.
2. **Reason / decide** — choose the next action, often expressed as a structured intention.
3. **Execute** — the orchestration layer runs the selected action.
4. **Append result** — add the tool output to the context.
5. **Re-read and decide again** — use the updated context to choose the next step.
6. **Check finishing condition** — stop when the task is complete or escalation is required.

This is commonly described as a **ReAct loop**: reason about the situation, act, observe the result, and repeat.

In [ ]:
# Conceptual pseudocode: a simplified agent orchestration loop.
# This is educational pseudocode, not a framework-specific implementation.

context = {"task": "Find the latest status of order 123"}

while True:
    intention = model_decide(context)       # Model proposes a structured next action

    if intention["type"] == "final_answer":
        if verify_success(intention, context):
            break
        context["verification_feedback"] = "Success criteria are not yet met."
        continue

    if intention["type"] == "human_approval":
        result = request_human_approval(intention)
    elif intention["type"] == "tool_call":
        result = orchestration_layer_execute(intention)
    else:
        result = {"error": "Unsupported action type"}

    context["last_result"] = result          # Result becomes part of the next model context

### 1.2 Why the orchestration layer matters
The model's proposed action is not itself execution. The wrapper can enforce:
- Tool allowlists and argument validation.
- Identity and authorization checks.
- Human approval for sensitive operations.
- Retry limits and timeout policies.
- Logging, result handling, and explicit termination.

**Example:** A support agent can read an account and draft a refund request, while the orchestration layer requires a human to approve the actual payment.

## 2. Four common action types
**Source: PDF page 4**

| Action | What it means | Example |
|---|---|---|
| **Call a tool** | Query a database, search documents, run code, or update a system | Search an order database |
| **Spawn a subagent** | Delegate a narrower subtask to another agent | Ask a policy agent to check refund eligibility |
| **Request human clarification** | Pause when the request is ambiguous, sensitive, or outside authority | Ask which account the user means |
| **Produce a final answer** | Return the completed result and signal the orchestration loop to end | Provide a verified order status |

These actions need explicit boundaries: task, scope, permissions, and available tools should be deliberate design choices.

### 2.1 Principle of Least Privilege for agents
Give an agent only the permissions needed for its role.

**Design example — refund workflow**
- Support agent: read order/account details and draft a refund.
- Policy tool: determine whether the request meets policy.
- Payment execution tool: unavailable to the support agent, or protected by an approval gate.
- Human approver: authorizes the irreversible payment action.

This limits the damage a mistaken model decision or compromised workflow can cause.

## 3. Model Context Protocol (MCP)
**Source: PDF page 5**

MCP standardizes how agents discover and use capabilities exposed by external systems. An MCP server can expose **Tools, Resources, and Prompts**, with descriptions that help the model understand when and how to use them.

### MCP interaction sequence
1. **Expose** — an MCP server publishes capabilities and descriptions.
2. **Discover** — the agent's MCP client discovers them through a consistent interface.
3. **Select** — the model selects a capability and provides structured arguments.
4. **Return** — the orchestration layer runs the operation and receives a structured result.
5. **Append** — the result is added to the agent's context.

Without a shared protocol, integrations with databases, file stores, and business services may each need custom interfaces.

**Important boundary:** MCP standardizes access; it does **not** replace authentication, authorization, input validation, or permission boundaries.

### 3.1 MCP in a practical example
Imagine a research agent has access to:
- A document-search tool.
- A company knowledge resource.
- A reusable research prompt.

The model can discover the available capabilities, choose document search, and provide a query. The orchestration layer invokes the capability and adds the returned passages to context. The agent then decides whether to search again or answer.

**Remember:** tool descriptions matter because the model uses them to decide which capability is appropriate.

## 4. Agent coordination and orchestration modes
**Source: PDF pages 6 and 10**

Choose a coordination pattern based on dependencies between tasks, whether agents work independently, and how control should move between them.

### 4.1 The five modes described in the source

| Mode | Control / information flow | Best fit | Main consideration |
|---|---|---|---|
| **Sequential** | One agent's output becomes the next agent's input | Ordered pipeline, such as research → draft → review | Later steps depend on earlier results |
| **Concurrent / Parallel** | Independent agents work simultaneously; results are merged | Independent searches or analyses | Merge and reconcile results |
| **Handoff** | One agent explicitly transfers control to a specific agent with relevant context | A clear transfer of responsibility | Context and ownership must transfer clearly |
| **Group Chat** | A manager coordinates agents in a shared conversation and selects who speaks next | Hub-and-spoke collaboration | Manager strategy and shared-thread complexity |
| **Magentic** | A manager dynamically creates/delegates work to subagents as needed | Open-ended, evolving tasks | More flexible, but harder to predict and control |

The source names these as orchestration modes from the Microsoft Agent Framework.

### 4.2 How to choose
- Use **sequential** when each stage needs the prior stage's output.
- Use **parallel** when tasks are independent and latency matters.
- Use **handoff** when control should move clearly from one specialist to another.
- Use **group chat** when a manager must coordinate multiple agents through a shared discussion.
- Use **Magentic** when the task is open-ended and the manager needs to decide dynamically what work to delegate.

These modes are not interchangeable: the structure of the task should drive the choice.

In [ ]:
# Example task decomposition (illustrative)
task = "Prepare a market-entry recommendation"

# Sequential: findings -> draft -> review
sequential = ["ResearchAgent", "WriterAgent", "ReviewerAgent"]

# Parallel: independent evidence gathering
parallel = ["CompetitorAgent", "CustomerAgent", "RegulationAgent"]

# Handoff: one agent transfers a case to a specialist
handoff = {"from": "SupportAgent", "to": "BillingSpecialist", "context": "Invoice dispute"}

# Group chat: manager selects which specialist speaks next
group_chat = {"manager": "CoordinatorAgent", "participants": ["FinanceAgent", "LegalAgent", "MarketAgent"]}

# Magentic: manager can dynamically decide which subagent/task is needed next
magentic = {"manager": "ResearchManager", "delegation": "dynamic"}

## 5. Retrieval-Augmented Generation (RAG) for agents
**Source: PDF page 7**

RAG retrieves relevant pieces of information (often document chunks) for a prompt and attaches them to the context used to generate the answer. The source highlights two use cases.

### 5.1 Classic RAG vs Agent RAG

| Dimension | Classic RAG | Agent RAG |
|---|---|---|
| Typical source | Relatively static corpus: policies, manuals, product documentation | Records of previous agent executions |
| Retrieved information | Relevant knowledge for the current question | Past transcripts, failures, decisions, approvals, and constraints |
| Main purpose | Ground a response in relevant reference material | Reuse operational experience and avoid repeating past mistakes |
| Example | Retrieve a refund policy | Retrieve a prior case where an API rejected a file above a size limit |

**Agent RAG is operational memory.** It does not mean the agent remembers everything. It retrieves relevant history after details have left the active context window.

### 5.2 Agent RAG workflow
1. Receive the current task.
2. Search the execution history for similar tasks, failures, decisions, approvals, or constraints.
3. Select relevant records.
4. Add the retrieved history to the active context.
5. Reassess the current task using that history.

**Design caution:** retrieved history is evidence to consider, not an instruction to blindly repeat an earlier decision. Validate whether the precedent still applies.

## 6. Stop conditions and error handling
**Source: PDF page 8**

An agent needs explicit rules for when to stop, retry, or ask for help. A plausible-sounding answer is not enough to prove the task is complete.

### 6.1 Stop / retry decision table

| Situation | Expected behavior | Example |
|---|---|---|
| **Success criteria met** | Stop only when the result is complete and verifiable | Required fields are present and status is confirmed |
| **Persistent error** | Escalate or give up when retrying will not help | Permission denied |
| **Temporary error** | Retry with a limit and backoff; after repeated failures, treat as persistent | Transient service timeout |
| **Lack of clarity** | Ask for missing information rather than inventing requirements | Two customer accounts match the name |
| **Safety / authority boundary reached** | Stop and request human intervention before an unauthorized or irreversible action | Approving a payment or blocking an account |

Human intervention is a safety valve, but it must be paired with clear finishing conditions and structured error paths.

### 6.2 Practical retry policy
A robust retry policy should define:
- Which errors are retryable.
- Maximum retry attempts.
- Backoff between attempts.
- A terminal state after the retry budget is exhausted.
- When to escalate to a person.
- What evidence is required to declare success.

Do not retry authorization failures indefinitely: a lack of permission is generally not a transient failure.

In [ ]:
# Illustrative policy logic
MAX_RETRIES = 3

def handle_failure(error, retry_count):
    if error.type == "temporary":
        if retry_count < MAX_RETRIES:
            return {"action": "retry_with_backoff"}
        return {"action": "escalate", "reason": "retry_limit_exceeded"}

    if error.type in {"permission_denied", "unsafe_action"}:
        return {"action": "stop_and_escalate"}

    if error.type == "missing_information":
        return {"action": "ask_user"}

    return {"action": "stop_and_report_error"}

## 7. Use cases from the source
**Source: PDF page 9**

### 7.1 Fraud triage swarm
- A manager coordinates transaction, customer-history, and risk specialists.
- It combines their findings into a structured fraud recommendation.
- Agents have narrow permissions.
- Human approval is required before freezing an account or blocking a payment.

**Concepts demonstrated:** manager coordination, specialist agents, least privilege, human approval.

### 7.2 Autonomous DevOps pipeline
- A manager spawns independent build and test agents in parallel.
- The pipeline proceeds only when results meet defined success criteria.
- The orchestration layer executes CI/CD calls, retries temporary failures, and stops or rolls back when errors persist.

**Concepts demonstrated:** parallel execution, verifiable success, retry policy, rollback/stop conditions.

### 7.3 Customer support memory system
- Agent RAG retrieves relevant tickets, decisions, approvals, and constraints.
- The current agent receives the history in its active context and reassesses the current request.
- It does not assume it remembers every prior conversation.

**Concepts demonstrated:** Agent RAG as operational memory.

### 7.4 Multi-vendor research collective
- Research agents discover tools through consistent interfaces.
- They exchange structured requests and results.
- Independent searches run concurrently; explicit handoffs are used when work depends on earlier findings.
- The system stops when verifiable success criteria are met.

**Concepts demonstrated:** MCP-style capability discovery, parallelism, handoffs, structured outputs.

## 8. End-to-end design checklist

Use this checklist when designing an agentic workflow:

1. **Define the task:** What outcome must be achieved?
2. **Define success:** What evidence makes completion verifiable?
3. **Choose coordination:** Sequential, parallel, handoff, group chat, or Magentic?
4. **Define tools:** Which capabilities are necessary and how are they described?
5. **Set permissions:** What can each agent read, write, approve, or execute?
6. **Define state/context:** What task details and tool results must be retained?
7. **Consider Agent RAG:** Would prior executions, failures, approvals, or constraints help?
8. **Handle errors:** Which errors are temporary, persistent, or caused by missing information?
9. **Set stop conditions:** When should the workflow finish, retry, or escalate?
10. **Require human approval:** Which actions are high-impact, irreversible, or outside the agent's authority?
11. **Log and evaluate:** Can you inspect decisions, tool results, retries, and the reason for termination?

## 9. Quick revision — key takeaways
**Source: PDF page 10**

1. **Decide, execute, repeat:** the model selects an action; orchestration executes it, adds results to context, and continues the loop.
2. **Bound every action:** tool calls, delegation, clarification, and final answers need explicit task, permission, and authority limits.
3. **Standardize tool access:** MCP makes capability discovery and tool interaction more consistent; security controls still apply.
4. **Match coordination to the task:** sequential for dependencies, parallel for independent work, handoff for transfer of control, manager patterns for dynamic coordination.
5. **Retrieve operational memory:** Agent RAG surfaces relevant prior executions and constraints.
6. **Stop by design:** define verified success, retryable failures, persistent errors, missing-information paths, and human-approval boundaries.

## 10. Self-test questions

Try answering these without looking at the notes. An answer key follows.

1. What is the difference between an LLM and an agentic system?
2. In the ReAct loop, who executes the tool call: the model or the orchestration layer?
3. Name the four common action types described in the source.
4. What does the principle of least privilege mean for agent design?
5. What does MCP standardize, and what does it **not** replace?
6. When is parallel coordination preferable to sequential coordination?
7. Explain the difference between handoff and group chat.
8. How does Magentic coordination differ from a fixed sequential pipeline?
9. What distinguishes Agent RAG from classic RAG?
10. How should a system handle a temporary timeout versus a permission-denied error?
11. Give two situations where the agent should ask for human intervention.
12. Why must success be verifiable rather than merely plausible?

### Answer key
1. An agentic system combines a model with orchestration that turns model decisions into actions and manages results and stopping.
2. The orchestration layer.
3. Call a tool; spawn a subagent; request human clarification; produce a final answer.
4. Give each agent only the permissions required for its assigned task.
5. MCP standardizes capability exposure/discovery and interaction. It does not replace authorization, validation, or permission boundaries.
6. When tasks are independent and their results can be merged.
7. Handoff explicitly transfers control to a particular agent; group chat uses a manager to coordinate agents in a shared conversation and select who speaks next.
8. Magentic uses a manager to delegate dynamically as needs emerge; a sequential pipeline follows a predetermined order.
9. Classic RAG retrieves reference knowledge; Agent RAG retrieves operational history such as prior executions, failures, decisions, approvals, and constraints.
10. Retry a temporary error within a bounded retry/backoff policy; stop or escalate a permission-denied error because retrying will not grant authority.
11. When information is ambiguous or missing, and before a sensitive/irreversible action outside the agent's authority.
12. Plausible language does not prove the task is complete; explicit success criteria need evidence.

## Source coverage map

| PDF pages | Topics covered here |
|---|---|
| 1 | Title and scope |
| 2–3 | Agent definition, orchestration, ReAct loop |
| 4 | Action types and least privilege |
| 5 | MCP |
| 6 | Coordination modes |
| 7 | Classic RAG and Agent RAG |
| 8 | Stop conditions and error handling |
| 9 | Four applied use cases |
| 10 | Key takeaways |

**Source:** *Foundations of LLM Agentic Systems*, supplied PDF (10 pages).